Un departamento de Recursos Humanos desea analizar la estructura de su personal y evaluar la representatividad de sus encuestas internas. Se te proporciona el archivo empleados_empresa.csv con $500$ registros.

In [2]:
## Lectura, limpieza e inspeccion
import pandas as pd
from pandas.api.types import CategoricalDtype

pd.options.display.float_format = '{:.2f}'.format

empleados = pd.read_csv("empleados_empresa.csv")
empleados = empleados.dropna().drop_duplicates()

print(empleados.shape)
print()
print(empleados.dtypes)


(500, 5)

empleado_id       str
departamento      str
rango_cargo       str
edad            int64
salario         int64
dtype: object


In [7]:
## Clasificacion de variables y tipado

empleados["departamento"] = empleados["departamento"].astype('category')

jerarquia_cargo = CategoricalDtype(
    categories=["Junior", "Semi-Senior", "Senior", "Director"],
    ordered=True
)

empleados["rango_cargo"] = empleados["rango_cargo"].astype(jerarquia_cargo)

print(empleados.dtypes)
#print(empleados[empleados["rango_cargo"] > "Semi-Senior"])


empleado_id          str
departamento    category
rango_cargo     category
edad               int64
salario            int64
dtype: object


In [12]:
# Muestreos

# MAS
muestra = 100
semilla = 2026

muestra_simple = empleados.sample(n=muestra, random_state=semilla)
#print(muestra_simple)

#Estratificado
frac = muestra / len(empleados)
muestra_estrat = empleados.groupby("rango_cargo", group_keys=False).sample(
    frac=frac,
    random_state=semilla
)
#print(muestra_estrat)

dist_poblacion = empleados["rango_cargo"].value_counts(normalize=True) * 100
dist_mas = muestra_simple["rango_cargo"].value_counts(normalize=True) * 100
dist_estrat = muestra_estrat["rango_cargo"].value_counts(normalize=True) * 100

comparacion_distribuciones = pd.DataFrame({
    "Poblacion (%)": dist_poblacion,
    "Muestreo Simple (%)": dist_mas,
    "Muestreo Estratificado (%)": dist_estrat
}).fillna(0).sort_index()

## Calculo el sesgo simple
comparacion_distribuciones["Sesgo Simple (%)"] = (
    comparacion_distribuciones["Muestreo Simple (%)"] - comparacion_distribuciones["Poblacion (%)"]
)

## Calculo el sesgo estratificado
comparacion_distribuciones["Sesgo Estratificado (%)"] = (
    comparacion_distribuciones["Muestreo Estratificado (%)"] - comparacion_distribuciones["Poblacion (%)"]
)


#Imprimo la tabla comparativa
print(comparacion_distribuciones.to_string())





             Poblacion (%)  Muestreo Simple (%)  Muestreo Estratificado (%)  Sesgo Simple (%)  Sesgo Estratificado (%)
rango_cargo                                                                                                           
Junior               37.60                35.00                       38.00             -2.60                     0.40
Semi-Senior          33.20                38.00                       33.00              4.80                    -0.20
Senior               20.00                15.00                       20.00             -5.00                     0.00
Director              9.20                12.00                        9.00              2.80                    -0.20
